In [65]:
# HYBRID MOVIE RECOMMENDATION SYSTEM
# TMDB (rich content metadata) + MovieLens (real user ratings)
# Content-Based + Collaborative Filtering + Offline Evaluation


import numpy as np
import pandas as pd
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

## Stage 1: Load All Data

This section loads the necessary datasets: TMDB (The Movie Database) for content metadata and MovieLens for user ratings. It then merges the TMDB credits with the TMDB movies dataset.

In [66]:
# TMDB (content metadata)
credits = pd.read_csv('/content/drive/MyDrive/Movie Recommendation System/tmdb_5000_credits.csv')
movies_tmdb = pd.read_csv('/content/drive/MyDrive/Movie Recommendation System/tmdb_5000_movies.csv')

# MovieLens (real user ratings)
ratings = pd.read_csv('/content/drive/MyDrive/Recommendation System/ratings.csv')
movies_ml = pd.read_csv('/content/drive/MyDrive/Recommendation System/movies.csv')
links = pd.read_csv('/content/drive/MyDrive/Recommendation System/links.csv')  # movieId -> tmdbId bridge

credits.drop(columns=['title'], inplace=True)
# Merge TMDB credits into TMDB movies
credits.columns = ['id', 'cast', 'crew']
movies_tmdb = movies_tmdb.merge(credits, on='id')

print("Data loaded successfully:")
print(f"TMDB Movies shape: {movies_tmdb.shape}")
print(f"MovieLens Ratings shape: {ratings.shape}")

Data loaded successfully:
TMDB Movies shape: (4803, 22)
MovieLens Ratings shape: (100836, 4)


## Stage 2: Bridge the Two Datasets via `links.csv`

`links.csv` provides a mapping between MovieLens `movieId` and TMDB `tmdbId`. This allows us to connect the rich content metadata from TMDB to the MovieLens `movieId`, which is used in the `ratings.csv`.

In [67]:
links['tmdbId'] = links['tmdbId'].astype('Int64')

bridge = links.merge(
    movies_tmdb, left_on='tmdbId', right_on='id', how='inner'
)

# bridge now has: movieId (MovieLens ID) + all TMDB content columns
print(f"Movies successfully linked across both datasets: {bridge.shape[0]} unique movies.")
print("Displaying the first few rows of the bridged dataframe:")
display(bridge.head())

Movies successfully linked across both datasets: 3537 unique movies.
Displaying the first few rows of the bridged dataframe:


,movieId,imdbId,tmdbId,budget,genres,homepage,id,keywords,original_language,original_title,...,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew
0,1,114709,862,30000000,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 35, ""...",http://toystory.disney.com/toy-story,862,"[{""id"": 931, ""name"": ""jealousy""}, {""id"": 4290,...",en,Toy Story,...,373554033,81.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,Toy Story,7.7,5269,"[{""cast_id"": 14, ""character"": ""Woody (voice)"",...","[{""credit_id"": ""52fe4284c3a36847f8024f55"", ""de..."
1,10,113189,710,58000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 28, ""...",http://www.mgm.com/view/movie/757/Goldeneye/,710,"[{""id"": 701, ""name"": ""cuba""}, {""id"": 769, ""nam...",en,GoldenEye,...,352194034,130.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,No limits. No fears. No substitutes.,GoldenEye,6.6,1174,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""52fe426ec3a36847f801e16f"", ""de..."
2,11,112346,9087,62000000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",NaN,9087,"[{""id"": 833, ""name"": ""white house""}, {""id"": 84...",en,The American President,...,107879496,106.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Why can't the most powerful man in the world h...,The American President,6.5,195,"[{""cast_id"": 1, ""character"": ""Andrew Shepherd""...","[{""credit_id"": ""52fe44dac3a36847f80adfa3"", ""de..."
3,14,113987,10858,44000000,"[{""id"": 36, ""name"": ""History""}, {""id"": 18, ""na...",NaN,10858,"[{""id"": 840, ""name"": ""usa president""}, {""id"": ...",en,Nixon,...,13681765,192.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Triumphant in Victory, Bitter in Defeat. He Ch...",Nixon,7.1,71,"[{""cast_id"": 1, ""character"": ""Richard Nixon"", ...","[{""credit_id"": ""52fe43c59251416c7501d705"", ""de..."
4,15,112760,1408,98000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",NaN,1408,"[{""id"": 911, ""name"": ""exotic island""}, {""id"": ...",en,Cutthroat Island,...,10017322,119.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,The Course Has Been Set. There Is No Turning B...,Cutthroat Island,5.7,136,"[{""cast_id"": 1, ""character"": ""Morgan Adams"", ""...","[{""credit_id"": ""52fe42f4c3a36847f802f69f"", ""de..."


In [68]:
# Filter ratings to only movies present in the bridge (fair comparison)
valid_movie_ids = set(bridge['movieId'].unique())
print(f"Ratings before filtering: {len(ratings)}")
ratings = ratings[ratings['movieId'].isin(valid_movie_ids)].reset_index(drop=True)
print(f"Ratings after filtering to TMDB-linked movies: {len(ratings)}")


Ratings before filtering: 100836
Ratings after filtering to TMDB-linked movies: 70194


## Stage 3: Build Content-Based Features (from TMDB metadata)

This stage extracts and processes relevant features from the TMDB metadata (cast, crew, keywords, genres, overview) to create a 'soup' of textual content for each movie. This soup will be used to calculate content similarity.

In [69]:
def get_director(x):
    for i in x:
        if i.get('job') == 'Director':
            return i['name']
    return ''

def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        return names[:3] if len(names) > 3 else names
    return []

def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    elif isinstance(x, str):
        return str.lower(x.replace(" ", ""))
    return ''

# Apply literal_eval to convert string representations of lists to actual lists
for feature in ['cast', 'crew', 'keywords', 'genres']:
    bridge[feature] = bridge[feature].apply(literal_eval)

# Extract director name
bridge['director'] = bridge['crew'].apply(get_director)

# Extract top 3 names for cast, keywords, and genres
for feature in ['cast', 'keywords', 'genres']:
    bridge[feature] = bridge[feature].apply(get_list)

# Clean (lowercase and remove spaces) the extracted data
for feature in ['cast', 'keywords', 'director', 'genres']:
    bridge[feature] = bridge[feature].apply(clean_data)

print("Content features (director, cast, keywords, genres) extracted and cleaned.")
display(bridge[['title', 'director', 'cast', 'keywords', 'genres']].head())

Content features (director, cast, keywords, genres) extracted and cleaned.


,title,director,cast,keywords,genres
0,Toy Story,johnlasseter,"[tomhanks, timallen, donrickles]","[jealousy, toy, boy]","[animation, comedy, family]"
1,GoldenEye,martincampbell,"[piercebrosnan, seanbean, izabellascorupco]","[cuba, falselyaccused, secretidentity]","[adventure, action, thriller]"
2,The American President,robreiner,"[michaeldouglas, annettebening, michaelj.fox]","[whitehouse, usapresident, newlove]","[comedy, drama, romance]"
3,Nixon,oliverstone,"[anthonyhopkins, joanallen, powersboothe]","[usapresident, presidentialelection, watergate...","[history, drama]"
4,Cutthroat Island,rennyharlin,"[geenadavis, matthewmodine, franklangella]","[exoticisland, treasure, map]","[action, adventure]"


In [70]:
def create_soup(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])

bridge['soup'] = bridge.apply(create_soup, axis=1)
bridge['overview'] = bridge['overview'].fillna('')

# Reset index so positional indices line up with movieId
bridge = bridge.reset_index(drop=True)
movieid_to_idx = pd.Series(bridge.index, index=bridge['movieId'])

print("Content 'soup' created and Movie ID to index mapping established.")
display(bridge[['title', 'soup']].head())

Content 'soup' created and Movie ID to index mapping established.


,title,soup
0,Toy Story,jealousy toy boy tomhanks timallen donrickles ...
1,GoldenEye,cuba falselyaccused secretidentity piercebrosn...
2,The American President,whitehouse usapresident newlove michaeldouglas...
3,Nixon,usapresident presidentialelection watergatesca...
4,Cutthroat Island,exoticisland treasure map geenadavis matthewmo...


### TF-IDF and Content Similarity

Using TF-IDF (Term Frequency-Inverse Document Frequency) on the movie overviews to quantify the importance of words and calculate the cosine similarity between movie overviews. This forms the basis for content-based recommendations.

In [71]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(bridge['overview'])
content_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

def content_based_recommend(movie_id, k=10):
    """Content-based recommendations using TMDB overview similarity."""
    if movie_id not in movieid_to_idx:
        print(f"Movie ID {movie_id} not found in the content dataset.")
        return []
    idx = movieid_to_idx[movie_id]
    sim_scores = list(enumerate(content_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:k + 1] # Exclude self
    result_idxs = [i[0] for i in sim_scores]
    return bridge['movieId'].iloc[result_idxs].tolist()

print("TF-IDF matrix for overview and content similarity computed.")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix for overview and content similarity computed.
TF-IDF matrix shape: (3537, 17633)


## Stage 4: Build Collaborative Filtering (from MovieLens ratings)

This stage sets up the infrastructure for collaborative filtering. It defines a function to create a sparse user-item matrix from the ratings data, which is essential for memory-efficient calculations with many users and items. It also includes functions for item-based K-Nearest Neighbors (KNN) to find similar movies and for recommending movies to a user based on their past highest-rated movies.

In [72]:
def create_X(df):
    """Builds a sparse user-item matrix from a ratings dataframe."""
    M = df['userId'].nunique()
    N = df['movieId'].nunique()

    user_mapper = dict(zip(np.unique(df["userId"]), list(range(M))))
    movie_mapper = dict(zip(np.unique(df["movieId"]), list(range(N))))
    user_inv_mapper = dict(zip(list(range(M)), np.unique(df["userId"])))
    movie_inv_mapper = dict(zip(list(range(N)), np.unique(df["movieId"])))

    user_index = [user_mapper[i] for i in df['userId']]
    item_index = [movie_mapper[i] for i in df['movieId']]

    X = csr_matrix((df["rating"], (user_index, item_index)), shape=(M, N))
    return X, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper

print("'create_X' function defined for building sparse user-item matrix.")

'create_X' function defined for building sparse user-item matrix.


In [73]:
# from sklearn.neighbors import NearestNeighbors

# def find_similar_movies(movie_id, X, movie_mapper, movie_inv_mapper, k=10, metric='cosine'):
#     """Item-based KNN collaborative filtering."""
#     X_T = X.T
#     if movie_id not in movie_mapper:
#         print(f"Movie ID {movie_id} not found in the collaborative filtering dataset.")
#         return []
#     movie_ind = movie_mapper[movie_id]
#     movie_vec = X_T[movie_ind]
#     if isinstance(movie_vec, np.ndarray):
#         movie_vec = movie_vec.reshape(1, -1)

#     kNN = NearestNeighbors(n_neighbors=min(k + 1, X_T.shape[0]), algorithm="brute", metric=metric)
#     kNN.fit(X_T)
#     neighbour = kNN.kneighbors(movie_vec, return_distance=False)

#     neighbour_ids = [movie_inv_mapper[neighbour.item(i)] for i in range(neighbour.shape[1])]
#     if movie_id in neighbour_ids:
#         neighbour_ids.remove(movie_id)
#     return neighbour_ids[:k]


# def collaborative_recommend_for_user(user_id, X, user_mapper, movie_mapper, movie_inv_mapper, k=10):
#     """Recommend movies for a user based on their highest-rated movie's neighbors."""
#     if user_id not in user_mapper:
#         print(f"User ID {user_id} not found in the collaborative filtering dataset.")
#         return []
#     user_idx = user_mapper[user_id]
#     user_ratings = X[user_idx].toarray().flatten()
#     rated = np.where(user_ratings > 0)[0]
#     if len(rated) == 0:
#         print(f"User {user_id} has no rated movies to base recommendations on.")
#         return []
#     # Find the movie with the highest rating for this user to use as a seed
#     seed_idx = rated[np.argmax(user_ratings[rated])]
#     seed_movie_id = movie_inv_mapper[seed_idx]
#     return find_similar_movies(seed_movie_id, X, movie_mapper, movie_inv_mapper, k=k)

# print("Collaborative filtering functions ('find_similar_movies', 'collaborative_recommend_for_user') defined.")



def train_svd(X, n_components=50):
    """
    Learns latent factors for users and items via truncated SVD.
    This captures global patterns across ALL interactions, rather than
    relying on nearest-neighbor lookups from a single seed item.
    """
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    user_factors = svd.fit_transform(X)      # shape: (n_users, n_components)
    item_factors = svd.components_.T          # shape: (n_items, n_components)
    return user_factors, item_factors


def svd_recommend_for_user(user_id, X, user_factors, item_factors, user_mapper, movie_inv_mapper, k=10):
    if user_id not in user_mapper:
        return []
    user_idx = user_mapper[user_id]
    scores = user_factors[user_idx] @ item_factors.T

    already_rated = X[user_idx].toarray().flatten() > 0
    scores = scores.copy()
    scores[already_rated] = -np.inf  # never recommend what they've already rated

    top_idxs = np.argsort(scores)[::-1][:k]
    return [movie_inv_mapper[i] for i in top_idxs]


## Stage 5: Hybrid Recommendation

This stage combines both content-based and collaborative filtering approaches. It generates recommendations from both methods and then blends them, giving a boosted score to movies that appear in both lists. This aims to leverage the strengths of both systems.

In [74]:
# def hybrid_recommend_for_user(user_id, X, user_mapper, movie_mapper, movie_inv_mapper,
#                                 k, content_weight, cf_weight):
#     """
#     Blends collaborative filtering candidates with content-based candidates.
#     Movies appearing in both lists get a combined boosted score.
#     """
#     cf_recs = collaborative_recommend_for_user(user_id, X, user_mapper, movie_mapper, movie_inv_mapper, k=k * 2)

#     # Use the user's top-rated movie as the seed for content-based candidates too
#     if user_id not in user_mapper:
#         return []
#     user_idx = user_mapper[user_id]
#     user_ratings = X[user_idx].toarray().flatten()
#     rated = np.where(user_ratings > 0)[0]
#     if len(rated) == 0:
#         return []
#     seed_idx = rated[np.argmax(user_ratings[rated])]
#     seed_movie_id = movie_inv_mapper[seed_idx]
#     content_recs = content_based_recommend(seed_movie_id, k=k * 2)

#     # Score: rank position converted to score, weighted sum
#     scores = {}
#     for rank, movie_id in enumerate(cf_recs):
#         scores[movie_id] = scores.get(movie_id, 0) + cf_weight * (1 / (rank + 1))
#     for rank, movie_id in enumerate(content_recs):
#         scores[movie_id] = scores.get(movie_id, 0) + content_weight * (1 / (rank + 1))

#     ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
#     return [movie_id for movie_id, _ in ranked[:k]]

# print("'hybrid_recommend_for_user' function defined for blending recommendations.")



# STAGE 6: HYBRID = SVD (candidates) + CONTENT (re-ranking boost)

def hybrid_recommend_for_user(user_id, X, user_factors, item_factors, user_mapper,
                                movie_mapper, movie_inv_mapper, k=10,
                                cf_weight=0.7, content_weight=0.3, n_candidates=30):
    """
    Uses SVD to generate a larger candidate pool, then re-ranks that pool
    by blending in content-based similarity to the user's favorite movie.
    Re-ranking a CF-generated pool (rather than merging two independent
    ranked lists) avoids injecting irrelevant content-based noise.
    """
    if user_id not in user_mapper:
        return []

    # Step 1: SVD generates a candidate pool (larger than final k)
    candidates = svd_recommend_for_user(
        user_id, X, user_factors, item_factors, user_mapper, movie_inv_mapper, k=n_candidates
    )
    if len(candidates) == 0:
        return []

    # Step 2: find user's favorite movie, to score candidates by content similarity to it
    user_idx = user_mapper[user_id]
    user_ratings = X[user_idx].toarray().flatten()
    rated = np.where(user_ratings > 0)[0]
    if len(rated) == 0:
        return candidates[:k]
    seed_idx = rated[np.argmax(user_ratings[rated])]
    seed_movie_id = movie_inv_mapper[seed_idx]

    if seed_movie_id not in movieid_to_idx:
        return candidates[:k]  # can't compute content similarity, fall back to pure CF

    seed_content_idx = movieid_to_idx[seed_movie_id]

    # Step 3: score each SVD candidate using: normalized CF rank + content similarity to seed
    scored_candidates = []
    for rank, movie_id in enumerate(candidates):
        cf_score = 1 - (rank / len(candidates))  # higher rank position = higher CF score
        if movie_id in movieid_to_idx:
            content_score = content_sim[seed_content_idx, movieid_to_idx[movie_id]]
        else:
            content_score = 0
        final_score = cf_weight * cf_score + content_weight * content_score
        scored_candidates.append((movie_id, final_score))

    scored_candidates.sort(key=lambda x: x[1], reverse=True)
    return [m for m, _ in scored_candidates[:k]]


## Stage 6: Offline Evaluation (Precision@K, Recall@K, NDCG@K)

This section defines the metrics and framework for evaluating the recommendation system offline. It includes functions to calculate Precision, Recall, and Normalized Discounted Cumulative Gain (NDCG) at a given K, and a generic evaluator that can be used for any recommendation function.

In [75]:
def precision_recall_ndcg_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = [1 if item in relevant else 0 for item in recommended_k]

    precision = sum(hits) / k if k > 0 else 0
    recall = sum(hits) / len(relevant) if len(relevant) > 0 else 0

    dcg = sum(hit / np.log2(idx + 2) for idx, hit in enumerate(hits)) # +2 because enumerate is 0-indexed
    ideal_hits = sorted(hits, reverse=True)
    idcg = sum(hit / np.log2(idx + 2) for idx, hit in enumerate(ideal_hits))
    ndcg = dcg / idcg if idcg > 0 else 0

    return precision, recall, ndcg


def evaluate_recommender(recommend_fn, train_ratings, test_ratings, k, relevance_threshold=4.0):
    """
    Generic evaluator: works for CF-only, content-only, or hybrid,
    as long as recommend_fn(user_id) -> list of movieIds is provided.
    """

    relevant_test = test_ratings[test_ratings['rating'] >= relevance_threshold]
    user_relevant_items = relevant_test.groupby('userId')['movieId'].apply(set).to_dict()

    precisions, recalls, ndcgs = [], [], []
    for user_id, relevant_items in user_relevant_items.items():
        recs = recommend_fn(user_id)
        if len(recs) == 0 or not relevant_items: # Skip if no recommendations or no relevant items
            continue
        p, r, n = precision_recall_ndcg_at_k(recs, relevant_items, k)
        precisions.append(p)
        recalls.append(r)
        ndcgs.append(n)

    if not precisions: # Handle case where no users were evaluated
        return {
            f"Precision@{k}": 0.0,
            f"Recall@{k}": 0.0,
            f"NDCG@{k}": 0.0,
            "n_users_evaluated": 0,
        }

    return {
        f"Precision@{k}": round(np.mean(precisions), 4),
        f"Recall@{k}": round(np.mean(recalls), 4),
        f"NDCG@{k}": round(np.mean(ndcgs), 4),
        "n_users_evaluated": len(precisions),
    }

print("Evaluation metrics and framework functions defined.")

Evaluation metrics and framework functions defined.


## Stage 7: Run the Full Experiment

This final stage orchestrates the entire experiment. It splits the MovieLens ratings into training and testing sets, builds the collaborative filtering model on the training data, and then evaluates both the pure collaborative filtering and the hybrid recommendation systems. Finally, it compares the performance of the hybrid model against the collaborative filtering baseline.

### Evaluate Collaborative Filtering (CF) Only

In [103]:
# K = 20

# # Split ratings into train/test
# train_ratings, test_ratings = train_test_split(
#     ratings, test_size=0.15, random_state=42
# )

# # Build the train-only user-item matrix (model must not see test interactions)
# X_train, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper = create_X(train_ratings)


K = 10

train_ratings, test_ratings = train_test_split(ratings, test_size=0.20, random_state=42)
X_train, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper = create_X(train_ratings)

print("\nTraining SVD model...")
user_factors, item_factors = train_svd(X_train, n_components=50)

print(f"Experiment K set to: {K}")
print(f"Train ratings shape: {train_ratings.shape}")
print(f"Test ratings shape: {test_ratings.shape}")
print(f"User-item matrix X_train shape: {X_train.shape}")


Training SVD model...
Experiment K set to: 10
Train ratings shape: (56155, 4)
Test ratings shape: (14039, 4)
User-item matrix X_train shape: (610, 3414)


In [104]:
# --- Baseline: SVD only ---
svd_only_fn = lambda uid: svd_recommend_for_user(
    uid, X_train, user_factors, item_factors, user_mapper, movie_inv_mapper, k=K
)
svd_results = evaluate_recommender(svd_only_fn,train_ratings, test_ratings, k=K)
print("SVD Matrix Factorization only:", svd_results)

SVD Matrix Factorization only: {'Precision@10': np.float64(0.1934), 'Recall@10': np.float64(0.2562), 'NDCG@10': np.float64(0.5341), 'n_users_evaluated': 591}


### Evaluate Hybrid Recommendation (Content + CF)

In [105]:
# --- Hybrid: SVD candidates + content-based re-ranking ---
hybrid_fn = lambda uid: hybrid_recommend_for_user(
    uid, X_train, user_factors, item_factors, user_mapper, movie_mapper, movie_inv_mapper,
    k=K, cf_weight=0.7, content_weight=0.3, n_candidates=30
)
hybrid_results = evaluate_recommender(hybrid_fn,train_ratings, test_ratings, k=K)
print("Hybrid (SVD + Content re-ranking):", hybrid_results)

Hybrid (SVD + Content re-ranking): {'Precision@10': np.float64(0.1937), 'Recall@10': np.float64(0.2564), 'NDCG@10': np.float64(0.5349), 'n_users_evaluated': 591}


### Compare Improvement over CF-only Baseline

In [106]:
print("\n--- Improvement over SVD-only baseline ---")
for metric in [f"Precision@{K}", f"Recall@{K}", f"NDCG@{K}"]:
    base_val = svd_results[metric]
    hybrid_val = hybrid_results[metric]
    if base_val > 0:
        pct_change = ((hybrid_val - base_val) / base_val) * 100
        print(f"{metric}: {base_val} -> {hybrid_val}  ({pct_change:+.1f}%)")
    else:
        print(f"{metric}: {base_val} -> {hybrid_val}")



--- Improvement over SVD-only baseline ---
Precision@10: 0.1934 -> 0.1937  (+0.2%)
Recall@10: 0.2562 -> 0.2564  (+0.1%)
NDCG@10: 0.5341 -> 0.5349  (+0.1%)


In [107]:
print("\n--- Weight sweep (finding best cf_weight/content_weight) ---")
best_result = None
for cf_w in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]:
    content_w = 1 - cf_w
    fn = lambda uid, cf_w=cf_w, content_w=content_w: hybrid_recommend_for_user(
        uid, X_train, user_factors, item_factors, user_mapper, movie_mapper, movie_inv_mapper,
        k=K, cf_weight=cf_w, content_weight=content_w, n_candidates=30
    )
    res = evaluate_recommender(fn, train_ratings,test_ratings, k=K)
    res['cf_weight'] = cf_w
    print(f"cf_weight={cf_w}, content_weight={content_w:.1f} -> {res}")
    if best_result is None or res[f"NDCG@{K}"] > best_result[f"NDCG@{K}"]:
        best_result = res

print("\nBest configuration found:", best_result)


--- Weight sweep (finding best cf_weight/content_weight) ---
cf_weight=1.0, content_weight=0.0 -> {'Precision@10': np.float64(0.1934), 'Recall@10': np.float64(0.2562), 'NDCG@10': np.float64(0.5341), 'n_users_evaluated': 591, 'cf_weight': 1.0}
cf_weight=0.9, content_weight=0.1 -> {'Precision@10': np.float64(0.1934), 'Recall@10': np.float64(0.2562), 'NDCG@10': np.float64(0.5348), 'n_users_evaluated': 591, 'cf_weight': 0.9}
cf_weight=0.8, content_weight=0.2 -> {'Precision@10': np.float64(0.1936), 'Recall@10': np.float64(0.2563), 'NDCG@10': np.float64(0.5352), 'n_users_evaluated': 591, 'cf_weight': 0.8}
cf_weight=0.7, content_weight=0.3 -> {'Precision@10': np.float64(0.1937), 'Recall@10': np.float64(0.2564), 'NDCG@10': np.float64(0.5349), 'n_users_evaluated': 591, 'cf_weight': 0.7}
cf_weight=0.6, content_weight=0.4 -> {'Precision@10': np.float64(0.1936), 'Recall@10': np.float64(0.2563), 'NDCG@10': np.float64(0.5359), 'n_users_evaluated': 591, 'cf_weight': 0.6}
cf_weight=0.5, content_weigh